# 04 Questions 3 and 4: Transfer Models

This notebook starts the transfer-analysis work by answering Q3:

**Can a cell-line-derived doxorubicin response signature be applied to TCGA-BRCA patient tumours and tested for association with survival?**

Q4 is not implemented in this PR. The focus here is one direction only: PRISM/DepMap cell-line response model -> TCGA-BRCA survival association.


## Conceptual Aim

DepMap/CCLE gives baseline gene expression for breast cancer cell lines. PRISM gives measured doxorubicin viability response for those same cell lines.

A simple regression model can learn a doxorubicin-sensitivity signature from the cell-line data:

```text
cell-line baseline expression -> PRISM doxorubicin sensitivity
```

The same gene coefficients can then be applied to TCGA-BRCA tumour expression to create a transferred doxorubicin-sensitivity-like score.

Important limitation: TCGA survival is not a direct doxorubicin-response endpoint. Any association with survival is prognostic or exploratory. It is not proof that the signature predicts chemotherapy benefit.


## Analysis Plan

1. Load the validated PRISM/DepMap breast cancer cell-line table.
2. Load the validated TCGA-BRCA survival and expression table.
3. Fit a small elastic-net regression model using shared p53/DNA-damage genes.
4. Save the learned gene coefficients.
5. Apply the same scaler and model to TCGA-BRCA tumours.
6. Test the transferred score with simple Cox regression.
7. Plot median high versus low Kaplan-Meier curves.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test
from scipy.stats import spearmanr
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

prism_path = project_root / "data/processed/brca_prism_doxorubicin_modelling_table.csv"
tcga_path = project_root / "data/processed/tcga_brca_survival_expression_table.csv"

coefficients_path = project_root / "tables/prism_doxorubicin_signature_coefficients.csv"
results_path = project_root / "tables/q3_prism_signature_tcga_survival_results.csv"
cellline_figure_path = project_root / "figures/q3_prism_signature_cellline_predicted_vs_observed.png"
cv_figure_path = project_root / "figures/q3_prism_signature_cellline_cv_predicted_vs_observed.png"
cv_results_path = project_root / "tables/q3_prism_signature_cross_validation_results.csv"
km_figure_path = project_root / "figures/q3_prism_signature_tcga_km.png"

for folder in [coefficients_path.parent, results_path.parent, cellline_figure_path.parent, cv_figure_path.parent, cv_results_path.parent, km_figure_path.parent]:
    folder.mkdir(parents=True, exist_ok=True)


In [ ]:
shared_genes = [
    "ATM", "CHEK2", "HIPK2", "MDM2", "PPM1D", "SIAH1", "TP53", "WSB1",
    "CDKN1A", "BAX", "BBC3", "GADD45A", "MDM4", "ATR", "CHEK1", "CASP3",
]

prism = pd.read_csv(prism_path)
tcga = pd.read_csv(tcga_path)

prism_missing = [gene for gene in shared_genes if gene not in prism.columns]
tcga_missing = [gene for gene in shared_genes if gene not in tcga.columns]

print("PRISM/DepMap table shape:", prism.shape)
print("TCGA-BRCA table shape:", tcga.shape)
print("Shared genes requested:", len(shared_genes))
print("Genes missing from PRISM/DepMap:", prism_missing)
print("Genes missing from TCGA-BRCA:", tcga_missing)

if prism_missing or tcga_missing:
    raise ValueError("A required shared gene is missing from one of the input tables.")

required_prism_columns = ["doxorubicin_sensitivity_score"] + shared_genes
required_tcga_columns = ["overall_survival_time", "overall_survival_event"] + shared_genes

prism_model_data = prism.dropna(subset=required_prism_columns).copy()
tcga_model_data = tcga.dropna(subset=required_tcga_columns).copy()

print("Cell lines available for Q3 signature fitting:", len(prism_model_data))
print("TCGA patients available before survival-model filtering:", len(tcga_model_data))

display(prism_model_data[["depmap_id", "cell_line_name", "doxorubicin_sensitivity_score"] + shared_genes[:4]].head())
display(tcga_model_data[["patient_id", "overall_survival_time", "overall_survival_event"] + shared_genes[:4]].head())


## Fit a Simple Cell-Line Doxorubicin Signature

The outcome is `doxorubicin_sensitivity_score`, where higher values mean greater doxorubicin sensitivity by construction.

Expression predictors are standardised before fitting. The scaler is fitted only on the cell-line data, then reused unchanged when scoring TCGA-BRCA tumours.

A small fixed elastic-net penalty is used (`alpha=0.1`, `l1_ratio=0.5`). This keeps the model simple and avoids selecting an all-zero signature in such a small dataset. Because there are only 26 breast cancer cell lines, this model is exploratory. The performance metrics below are apparent in-sample checks, not a strong estimate of future predictive accuracy.


In [ ]:
X_cell = prism_model_data[shared_genes].astype(float)
y_cell = prism_model_data["doxorubicin_sensitivity_score"].astype(float)

scaler = StandardScaler()
X_cell_scaled = scaler.fit_transform(X_cell)

elastic_net = ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000, random_state=0)
elastic_net.fit(X_cell_scaled, y_cell)

cell_predicted = elastic_net.predict(X_cell_scaled)

r2 = r2_score(y_cell, cell_predicted)
rmse = np.sqrt(mean_squared_error(y_cell, cell_predicted))
spearman_r, spearman_p = spearmanr(y_cell, cell_predicted)

print("Elastic-net alpha:", elastic_net.alpha)
print("Elastic-net l1_ratio:", elastic_net.l1_ratio)
print("Apparent in-sample R^2:", round(r2, 3))
print("Apparent in-sample RMSE:", round(rmse, 3))
print("Spearman correlation:", round(spearman_r, 3))
print("Spearman p-value:", spearman_p)


In [ ]:
coefficient_table = pd.DataFrame({
    "gene": shared_genes,
    "coefficient": elastic_net.coef_,
})
coefficient_table["absolute_coefficient"] = coefficient_table["coefficient"].abs()
coefficient_table["non_zero"] = coefficient_table["coefficient"] != 0
coefficient_table["model_intercept"] = elastic_net.intercept_
coefficient_table["model_alpha"] = elastic_net.alpha
coefficient_table["model_l1_ratio"] = elastic_net.l1_ratio
coefficient_table = coefficient_table.sort_values("absolute_coefficient", ascending=False)

coefficient_table.to_csv(coefficients_path, index=False)

non_zero_genes = coefficient_table.loc[coefficient_table["non_zero"], "gene"].tolist()
print("Non-zero coefficient genes:", non_zero_genes)
print("Saved coefficients to:", coefficients_path.relative_to(project_root))
display(coefficient_table)


In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(y_cell, cell_predicted, color="#2f6f9f", edgecolor="black", alpha=0.85)

plot_min = min(y_cell.min(), cell_predicted.min())
plot_max = max(y_cell.max(), cell_predicted.max())
plt.plot([plot_min, plot_max], [plot_min, plot_max], color="black", linestyle="--", linewidth=1)

plt.xlabel("Observed PRISM doxorubicin sensitivity score")
plt.ylabel("Predicted sensitivity score")
plt.title("Cell-line doxorubicin signature fit")
plt.text(
    0.03,
    0.97,
    f"n = {len(y_cell)}\nR^2 = {r2:.2f}\nRMSE = {rmse:.2f}\nSpearman r = {spearman_r:.2f}",
    transform=plt.gca().transAxes,
    va="top",
    bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "edgecolor": "lightgray"},
)
plt.tight_layout()
plt.savefig(cellline_figure_path, dpi=200)
plt.show()

print("Saved figure to:", cellline_figure_path.relative_to(project_root))


## Cross-validation sensitivity check

The model fit above is apparent/in-sample: it evaluates predictions on the same 26 cell lines used to fit the model.

To get a more realistic, although still unstable, estimate of generalisation, this section uses leave-one-out cross-validation. Each fold leaves out one breast cancer cell line, fits the scaler and elastic-net model on the other 25 cell lines, then predicts the held-out cell line.

Because there are only 26 breast cancer cell lines, the cross-validated metrics can move around substantially and should be interpreted cautiously. The TCGA survival transfer result remains the primary Q3 clinical test.


In [ ]:
loo = LeaveOneOut()
cv_predictions = []

for train_index, test_index in loo.split(X_cell):
    X_train = X_cell.iloc[train_index]
    X_test = X_cell.iloc[test_index]
    y_train = y_cell.iloc[train_index]

    fold_scaler = StandardScaler()
    X_train_scaled = fold_scaler.fit_transform(X_train)
    X_test_scaled = fold_scaler.transform(X_test)

    fold_model = ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000, random_state=0)
    fold_model.fit(X_train_scaled, y_train)

    cv_predictions.append(float(fold_model.predict(X_test_scaled)[0]))

cv_predictions = np.array(cv_predictions)

cv_r2 = r2_score(y_cell, cv_predictions)
cv_rmse = np.sqrt(mean_squared_error(y_cell, cv_predictions))
cv_spearman_r, cv_spearman_p = spearmanr(y_cell, cv_predictions)

cv_results = pd.DataFrame([
    {"metric": "n_cell_lines", "value": len(y_cell)},
    {"metric": "cross_validation", "value": "leave-one-out"},
    {"metric": "model", "value": "ElasticNet(alpha=0.1, l1_ratio=0.5)"},
    {"metric": "cv_r2", "value": cv_r2},
    {"metric": "cv_rmse", "value": cv_rmse},
    {"metric": "cv_spearman_r", "value": cv_spearman_r},
    {"metric": "cv_spearman_p", "value": cv_spearman_p},
])
cv_results.to_csv(cv_results_path, index=False)

cv_prediction_table = prism_model_data[["depmap_id", "cell_line_name", "doxorubicin_sensitivity_score"]].copy()
cv_prediction_table["cv_predicted_sensitivity_score"] = cv_predictions

print("Leave-one-out CV R^2:", round(cv_r2, 3))
print("Leave-one-out CV RMSE:", round(cv_rmse, 3))
print("Leave-one-out CV Spearman r:", round(cv_spearman_r, 3))
print("Leave-one-out CV Spearman p-value:", cv_spearman_p)
print("Saved CV results to:", cv_results_path.relative_to(project_root))

display(cv_results)
display(cv_prediction_table.head())


In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(y_cell, cv_predictions, color="#8c4b7d", edgecolor="black", alpha=0.85)

plot_min = min(y_cell.min(), cv_predictions.min())
plot_max = max(y_cell.max(), cv_predictions.max())
plt.plot([plot_min, plot_max], [plot_min, plot_max], color="black", linestyle="--", linewidth=1)

plt.xlabel("Observed PRISM doxorubicin sensitivity score")
plt.ylabel("Leave-one-out predicted sensitivity score")
plt.title("Cross-validated cell-line doxorubicin signature")
plt.text(
    0.03,
    0.97,
    f"n = {len(y_cell)}\nLOOCV R^2 = {cv_r2:.2f}\nLOOCV RMSE = {cv_rmse:.2f}\nSpearman r = {cv_spearman_r:.2f}",
    transform=plt.gca().transAxes,
    va="top",
    bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "edgecolor": "lightgray"},
)
plt.tight_layout()
plt.savefig(cv_figure_path, dpi=200)
plt.show()

print("Saved cross-validation figure to:", cv_figure_path.relative_to(project_root))


## Apply the Cell-Line Signature to TCGA-BRCA

The TCGA score is calculated by applying the cell-line-fitted scaler and elastic-net model to TCGA tumour expression for the same genes.

This gives a doxorubicin-sensitivity-like score in patients. It should be interpreted as a transferred expression signature score, not as measured TCGA drug response.


In [ ]:
X_tcga = tcga_model_data[shared_genes].astype(float)
X_tcga_scaled = scaler.transform(X_tcga)

tcga_scored = tcga_model_data.copy()
tcga_scored["prism_doxorubicin_signature_score"] = elastic_net.predict(X_tcga_scaled)

survival_columns = [
    "patient_id",
    "overall_survival_time",
    "overall_survival_event",
    "prism_doxorubicin_signature_score",
]
if "age" in tcga_scored.columns:
    survival_columns.append("age")
if "stage" in tcga_scored.columns:
    survival_columns.append("stage")

survival_data = tcga_scored[survival_columns].copy()
survival_data["overall_survival_time"] = pd.to_numeric(survival_data["overall_survival_time"], errors="coerce")
survival_data["overall_survival_event"] = pd.to_numeric(survival_data["overall_survival_event"], errors="coerce")

if "age" in survival_data.columns:
    survival_data["age"] = pd.to_numeric(survival_data["age"], errors="coerce")

survival_data = survival_data.dropna(subset=[
    "overall_survival_time",
    "overall_survival_event",
    "prism_doxorubicin_signature_score",
]).copy()

# Cox and Kaplan-Meier models require a positive survival time.
survival_data = survival_data[survival_data["overall_survival_time"] > 0].copy()
survival_data["overall_survival_event"] = survival_data["overall_survival_event"].astype(int)

print("TCGA patients available for transferred-score survival testing:", len(survival_data))
print("Deaths/events:", int((survival_data["overall_survival_event"] == 1).sum()))
print("Censored:", int((survival_data["overall_survival_event"] == 0).sum()))
print("Signature score range:", survival_data["prism_doxorubicin_signature_score"].min(), survival_data["prism_doxorubicin_signature_score"].max())

display(survival_data.head())


## Test the Transferred Score Against TCGA Survival

A Cox model tests whether the continuous transferred score is associated with overall survival.

The first model uses only the transferred signature score. The second model adjusts for age when age is available.

Stage is not included here because TCGA stage labels are categorical and not yet cleaned for a simple numeric model. It can be added later as a sensitivity analysis.


In [ ]:
cox_result_rows = []

def add_cox_results(model_name, cox_model, model_data):
    summary = cox_model.summary.reset_index().rename(columns={"covariate": "term"})
    for _, row in summary.iterrows():
        cox_result_rows.append({
            "analysis": model_name,
            "term": row["term"],
            "coef": row["coef"],
            "hazard_ratio": row["exp(coef)"],
            "ci_lower": row["exp(coef) lower 95%"],
            "ci_upper": row["exp(coef) upper 95%"],
            "p_value": row["p"],
            "n_patients": len(model_data),
            "n_events": int(model_data["overall_survival_event"].sum()),
        })

cox_unadjusted_data = survival_data[[
    "overall_survival_time",
    "overall_survival_event",
    "prism_doxorubicin_signature_score",
]].copy()

cox_unadjusted = CoxPHFitter()
cox_unadjusted.fit(
    cox_unadjusted_data,
    duration_col="overall_survival_time",
    event_col="overall_survival_event",
)
add_cox_results("cox_signature_only", cox_unadjusted, cox_unadjusted_data)

print("Unadjusted Cox model")
display(cox_unadjusted.summary)

if "age" in survival_data.columns and survival_data["age"].notna().sum() > 0:
    cox_age_data = survival_data[[
        "overall_survival_time",
        "overall_survival_event",
        "prism_doxorubicin_signature_score",
        "age",
    ]].dropna().copy()

    cox_age_adjusted = CoxPHFitter()
    cox_age_adjusted.fit(
        cox_age_data,
        duration_col="overall_survival_time",
        event_col="overall_survival_event",
    )
    add_cox_results("cox_signature_plus_age", cox_age_adjusted, cox_age_data)

    print("Age-adjusted Cox model")
    display(cox_age_adjusted.summary)
else:
    print("Age-adjusted Cox model skipped because age is unavailable or fully missing.")


In [ ]:
score_median = survival_data["prism_doxorubicin_signature_score"].median()
survival_data["signature_group"] = np.where(
    survival_data["prism_doxorubicin_signature_score"] >= score_median,
    "High transferred score",
    "Low transferred score",
)

high_group = survival_data[survival_data["signature_group"] == "High transferred score"]
low_group = survival_data[survival_data["signature_group"] == "Low transferred score"]

logrank = logrank_test(
    high_group["overall_survival_time"],
    low_group["overall_survival_time"],
    event_observed_A=high_group["overall_survival_event"],
    event_observed_B=low_group["overall_survival_event"],
)

cox_result_rows.append({
    "analysis": "median_high_vs_low_logrank",
    "term": "High transferred score vs low transferred score",
    "coef": np.nan,
    "hazard_ratio": np.nan,
    "ci_lower": np.nan,
    "ci_upper": np.nan,
    "p_value": logrank.p_value,
    "n_patients": len(survival_data),
    "n_events": int(survival_data["overall_survival_event"].sum()),
})

results_table = pd.DataFrame(cox_result_rows)
results_table.to_csv(results_path, index=False)

print("Median transferred score:", score_median)
print("High-score patients:", len(high_group))
print("Low-score patients:", len(low_group))
print("Log-rank p-value:", logrank.p_value)
print("Saved survival results to:", results_path.relative_to(project_root))
display(results_table)


In [ ]:
kmf = KaplanMeierFitter()

plt.figure(figsize=(7, 5))
for group_name, group_data in survival_data.groupby("signature_group"):
    kmf.fit(
        durations=group_data["overall_survival_time"],
        event_observed=group_data["overall_survival_event"],
        label=f"{group_name} (n={len(group_data)})",
    )
    kmf.plot_survival_function(ci_show=False)

plt.xlabel("Overall survival time (days)")
plt.ylabel("Survival probability")
plt.title("TCGA-BRCA survival by transferred PRISM doxorubicin signature")
plt.text(
    0.03,
    0.08,
    f"Median split\nLog-rank p = {logrank.p_value:.3g}",
    transform=plt.gca().transAxes,
    bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "edgecolor": "lightgray"},
)
plt.tight_layout()
plt.savefig(km_figure_path, dpi=200)
plt.show()

print("Saved Kaplan-Meier figure to:", km_figure_path.relative_to(project_root))


## Short Interpretation

The coefficient table shows which p53/DNA-damage genes contribute to the exploratory PRISM doxorubicin response signature. Genes with zero coefficients were not used by the fitted elastic-net model.

The observed-versus-predicted plot checks how well the fitted signature explains doxorubicin sensitivity within the 26 breast cancer cell lines. Because this is a very small training set, this should be treated as an exploratory apparent fit, not a robust external validation.

The leave-one-out cross-validation section gives a more realistic sensitivity check because each prediction is made for a held-out cell line. These cross-validated metrics are still unstable because there are only 26 breast cancer cell lines.

The TCGA Cox and Kaplan-Meier analyses test whether the transferred cell-line-derived score is associated with patient overall survival. This is a prognostic survival association. TCGA-BRCA does not directly measure doxorubicin response, so this analysis cannot prove that the score predicts doxorubicin benefit.

Stage was not included in this first model because the stage labels need separate cleaning before they can be used safely in a simple Cox model.
